# Skin Lesion Ensemble — DenseNet121 + EfficientNet-B3 + DINOv2

Built on top of the three baseline notebooks. Reuses each notebook's saved
`predictions.csv` (test) and `val_predictions.csv` (validation) instead of
reloading checkpoints — all three notebooks already write these with an
identical, verified schema (`image_id, true_label, predicted_label,
probability_benign, probability_malignant`), aligned to `test_df`/`val_df`
by `image_id` and de-duplicated. That means no architecture code needs to
be re-implemented here; the ensemble operates purely on saved probabilities.

============================================================================
AUDIT — what was checked in the three notebooks, and what this script fixes
============================================================================

1. MALIGNANT RECALL (the metric the assignment worried was missing/wrong)
   All three notebooks already call `recall_score(labels, preds,
   zero_division=0)` with sklearn's default `pos_label=1`, and label 1 =
   Malignant everywhere (`probability_malignant` is the positive-class
   probability in all three). So the existing per-model "recall" IS already
   Malignant recall, not micro recall — this part was correct. Same for the
   existing `precision` and `f1_score` calls (binary averaging, pos_label=1
   default → Malignant precision/F1). None of the three notebooks compute
   Accuracy, Micro-F1, or Micro-Recall as a *primary* metric already, so
   there was no "picked the wrong headline metric" bug either.

2. REAL GAPS FOUND (fixed below):
   - DenseNet121 notebook never imports/calls `average_precision_score` —
     it has no PR-AUC at all, unlike the other two. Fixed: PR-AUC computed
     for every model + every ensemble here, uniformly.
   - None of the three notebooks compute Specificity, Macro-F1, Micro-F1,
     Micro-Recall, or explicit False-Negative-Rate. Added below for all
     models and all ensemble variants, per your spec.
   - `confusion_matrix(...)` is called without `labels=[0, 1]` in all three
     notebooks. With clean {0,1} integer labels this happens to sort
     correctly today, but it's implicit. Fixed: every confusion matrix call
     in this script passes `labels=[0, 1]` explicitly.
   - DenseNet's `predicted_label` column in predictions.csv is baked in at
     its own F1-optimal threshold (0.41); DINOv2/EfficientNet-B3 similarly
     bake in their own per-model thresholds. That's fine for *reporting
     each model standalone*, but it is NOT reused for the ensemble — the
     ensemble here works from `probability_malignant` (continuous) and
     re-thresholds independently, per Section 4 below. Each model's own
     `predicted_label` is only used for the "individual model" rows of the
     comparison table, computed the same way as the ensemble for
     consistency (metrics recomputed from probability + a threshold chosen
     the same way), not taken as pre-computed.

3. THRESHOLD / LEAKAGE METHODOLOGY — already correct, kept as-is
   All three notebooks already: (a) tune their threshold on the validation
   set only, (b) freeze it, (c) evaluate the test set once. DenseNet's
   docstring even documents that its *earlier* version used to test
   threshold 0.5/0.3 directly on the test set and that this V2 fixes it.
   Calibration (Platt scaling) is also fit on validation only in all three.
   This script follows the same discipline for the ensemble weights and
   the ensemble threshold: both are chosen on validation predictions only;
   the test predictions are loaded but never touched until Section 8.

============================================================================


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, roc_curve, precision_recall_curve,
)

POSITIVE_CLASS = 1  # Malignant. 0 = Benign. Never swap this.

## 0. Paths — point these at each notebook's `predictions/` folder

In [ ]:
MODEL_DIRS = {
    "DenseNet121":     Path(r"D:\SkinCancer_Project\outputs\binary\Binary_DenseNet121_Baseline_V2\predictions"),
    "EfficientNet-B3": Path(r"D:\SkinCancer_Project\outputs\binary\Binary_EfficientNetB3_Baseline_V2\predictions"),
    "DINOv2":          Path(r"D:\SkinCancer_Project\outputs\binary\Binary_DINOv2_Baseline_V2\predictions"),
}
OUTPUT_DIR = Path(r"D:\SkinCancer_Project\outputs\binary\Ensemble_V1")
for sub in ["", "plots", "predictions"]:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

REQUIRED_COLS = {"image_id", "true_label", "probability_benign", "probability_malignant"}

## 1. Load + validate — Step 2 checks (compatible labels, splits, no leakage)

In [ ]:
def load_split(split_name: str) -> dict[str, pd.DataFrame]:
    """split_name: 'predictions' (test) or 'val_predictions'."""
    frames = {}
    for name, d in MODEL_DIRS.items():
        df = pd.read_csv(d / f"{split_name}.csv")
        missing = REQUIRED_COLS - set(df.columns)
        if missing:
            raise RuntimeError(f"{name}/{split_name}.csv missing columns: {missing}")
        if df["image_id"].duplicated().any():
            raise RuntimeError(f"{name}/{split_name}.csv has duplicate image_id values")
        frames[name] = df.set_index("image_id").sort_index()
    return frames

def check_alignment(frames: dict[str, pd.DataFrame], split_name: str):
    names = list(frames)
    ref_ids = set(frames[names[0]].index)
    for n in names[1:]:
        ids = set(frames[n].index)
        if ids != ref_ids:
            raise RuntimeError(
                f"[{split_name}] {n} covers different samples than {names[0]}: "
                f"missing={len(ref_ids - ids)}, extra={len(ids - ref_ids)}. "
                "Cannot ensemble — the three notebooks must share identical val/test splits."
            )
    ref_labels = frames[names[0]]["true_label"]
    for n in names[1:]:
        if not frames[n].loc[ref_labels.index, "true_label"].equals(ref_labels):
            raise RuntimeError(f"[{split_name}] {n} has different true_label values for the same image_id — "
                                "label mapping is inconsistent across notebooks.")
    print(f"[{split_name}] OK — {len(ref_ids)} samples, identical ids and labels across all 3 models.")

val_frames = load_split("val_predictions")
test_frames = load_split("predictions")
check_alignment(val_frames, "validation")
check_alignment(test_frames, "test")

# Leakage check: no image_id should appear in both val and test for any model
overlap = set(val_frames["DenseNet121"].index) & set(test_frames["DenseNet121"].index)
if overlap:
    raise RuntimeError(f"Validation/test leakage: {len(overlap)} image_id(s) appear in both splits: "
                        f"{list(overlap)[:5]}")
print("No val/test image_id overlap.")

def build_matrix(frames: dict[str, pd.DataFrame]) -> pd.DataFrame:
    names = list(frames)
    out = pd.DataFrame(index=frames[names[0]].index)
    out["true_label"] = frames[names[0]]["true_label"].astype(int)
    for n in names:
        out[f"{n}_prob"] = frames[n]["probability_malignant"].astype(float)
    return out

val_mat = build_matrix(val_frames)
test_mat = build_matrix(test_frames)
MODEL_NAMES = list(MODEL_DIRS)

## 2. Metric block — every metric from the spec, computed correctly and consistently

In [ ]:
def compute_metrics(y_true, y_prob, threshold: float) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else float("nan")   # = recall_score(pos_label=1)
    specificity = tn / (tn + fp) if (tn + fp) else float("nan")
    fnr = fn / (tp + fn) if (tp + fn) else float("nan")
    bal_acc_manual = (sensitivity + specificity) / 2

    sk_recall_pos1 = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    sk_bal_acc = balanced_accuracy_score(y_true, y_pred)
    # Sanity cross-check (Section 10 of the spec) — must match sklearn to float tolerance.
    assert abs(sensitivity - sk_recall_pos1) < 1e-9, "sensitivity != sklearn recall_score(pos_label=1)"
    assert abs(bal_acc_manual - sk_bal_acc) < 1e-9, "manual balanced accuracy != sklearn balanced_accuracy_score"

    return {
        "threshold": threshold,
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "malignant_recall_sensitivity": sensitivity,
        "false_negative_rate": fnr,
        "specificity": specificity,
        "balanced_accuracy": sk_bal_acc,
        "malignant_precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "malignant_f1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "benign_f1": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "micro_f1": f1_score(y_true, y_pred, average="micro", zero_division=0),          # supplementary only
        "micro_recall": recall_score(y_true, y_pred, average="micro", zero_division=0),  # supplementary only
        "accuracy": accuracy_score(y_true, y_pred),                                       # supplementary only
        "roc_auc": roc_auc_score(y_true, y_prob),          # from probabilities, not y_pred
        "pr_auc": average_precision_score(y_true, y_prob),  # from probabilities, not y_pred
    }

## 3. Complementarity analysis (Step 4) — done BEFORE trusting an ensemble helps

In [ ]:
prob_cols = [f"{n}_prob" for n in MODEL_NAMES]
corr = val_mat[prob_cols].corr()
print("Validation-set pairwise probability correlation:\n", corr.round(3))

val_preds_05 = {n: (val_mat[f"{n}_prob"] >= 0.5).astype(int) for n in MODEL_NAMES}
disagreement = pd.DataFrame(index=MODEL_NAMES, columns=MODEL_NAMES, dtype=float)
for a in MODEL_NAMES:
    for b in MODEL_NAMES:
        disagreement.loc[a, b] = (val_preds_05[a] != val_preds_05[b]).mean()
print("\nPairwise disagreement rate (@0.5, validation):\n", disagreement.round(3))

# Who is uniquely correct where the others fail? (identifies complementary error patterns)
y_val = val_mat["true_label"].values
unique_correct = {}
for n in MODEL_NAMES:
    others = [m for m in MODEL_NAMES if m != n]
    this_correct = val_preds_05[n].values == y_val
    others_wrong = np.all([val_preds_05[o].values != y_val for o in others], axis=0)
    unique_correct[n] = int((this_correct & others_wrong).sum())
print("\nSamples where only this model is correct (@0.5, validation):", unique_correct)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=0, vmax=1, ax=ax)
ax.set_title("Validation-set probability correlation")
fig.savefig(OUTPUT_DIR / "plots" / "prediction_correlation_heatmap.png", bbox_inches="tight", dpi=150)
plt.close(fig)

## 4. Ensemble weight optimization — VALIDATION SET ONLY (Step 5)

Objective is NOT accuracy/micro-F1. Primary: Malignant sensitivity +
balanced accuracy; secondary: PR-AUC. A fixed 0.5 cutoff is used only
*inside* this search to score candidate weights consistently — the actual
operating threshold is optimized separately in Section 5 after the weights
are frozen.

In [ ]:
def ensemble_prob(mat: pd.DataFrame, weights: np.ndarray) -> np.ndarray:
    return sum(w * mat[f"{n}_prob"].values for n, w in zip(MODEL_NAMES, weights))

def neg_objective(raw_weights: np.ndarray) -> float:
    w = np.abs(raw_weights)
    w = w / w.sum()
    p = ensemble_prob(val_mat, w)
    m = compute_metrics(y_val, p, threshold=0.5)
    # primary: sensitivity + balanced accuracy; secondary: PR-AUC (small weight, tie-breaker only)
    score = 0.45 * m["malignant_recall_sensitivity"] + 0.45 * m["balanced_accuracy"] + 0.10 * m["pr_auc"]
    return -score

best = None
rng = np.random.default_rng(0)
for _ in range(25):  # multi-start to avoid a bad local optimum
    x0 = rng.dirichlet(np.ones(len(MODEL_NAMES)))
    res = minimize(neg_objective, x0, method="Nelder-Mead",
                    options={"xatol": 1e-4, "fatol": 1e-6, "maxiter": 2000})
    if best is None or res.fun < best.fun:
        best = res

optimal_weights = np.abs(best.x)
optimal_weights = optimal_weights / optimal_weights.sum()
weights_dict = dict(zip(MODEL_NAMES, optimal_weights.tolist()))
print("Optimized weights (validation only):", {k: round(v, 3) for k, v in weights_dict.items()})

equal_weights = np.ones(len(MODEL_NAMES)) / len(MODEL_NAMES)

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(MODEL_NAMES, optimal_weights)
ax.set_ylabel("Ensemble weight")
ax.set_title("Optimized ensemble weights (validation-selected)")
fig.savefig(OUTPUT_DIR / "plots" / "ensemble_weights.png", bbox_inches="tight", dpi=150)
plt.close(fig)

## 5. Threshold optimization — VALIDATION SET ONLY (Step 6)
Objective: prioritize sensitivity, then balanced accuracy, subject to a
minimum acceptable specificity floor — matches the spec's stated priority
order (1. sensitivity, 2. balanced accuracy, 3. acceptable specificity).

In [ ]:
MIN_SPECIFICITY = 0.60  # adjust to your project's clinical tolerance for false positives

def select_threshold(y_true, y_prob) -> tuple[float, pd.DataFrame]:
    rows = []
    for t in np.arange(0.05, 0.951, 0.01):
        rows.append(compute_metrics(y_true, y_prob, t))
    sweep = pd.DataFrame(rows)
    feasible = sweep[sweep["specificity"] >= MIN_SPECIFICITY]
    pool = feasible if len(feasible) else sweep  # fall back if nothing clears the floor
    chosen = pool.sort_values(
        ["malignant_recall_sensitivity", "balanced_accuracy"], ascending=False
    ).iloc[0]
    return float(chosen["threshold"]), sweep

val_ensemble_prob_opt = ensemble_prob(val_mat, optimal_weights)
best_threshold, threshold_sweep = select_threshold(y_val, val_ensemble_prob_opt)
print(f"Selected ensemble threshold (validation only): {best_threshold:.2f}")

fig, ax = plt.subplots(figsize=(6, 4))
for col, label in [("malignant_recall_sensitivity", "Sensitivity"),
                    ("specificity", "Specificity"),
                    ("balanced_accuracy", "Balanced Acc."),
                    ("malignant_f1", "Malignant F1")]:
    ax.plot(threshold_sweep["threshold"], threshold_sweep[col], label=label)
ax.axvline(best_threshold, color="red", linestyle="--", label=f"chosen = {best_threshold:.2f}")
ax.set_xlabel("Decision threshold"); ax.set_ylabel("Score"); ax.legend()
ax.set_title("Threshold vs metrics — validation set (ensemble)")
fig.savefig(OUTPUT_DIR / "plots" / "threshold_vs_metrics.png", bbox_inches="tight", dpi=150)
plt.close(fig)

## 6. Score every model + every ensemble variant, same way, on validation
(for the comparison table's validation columns and for picking each
individual model's own threshold the same way the ensemble's was picked —
not reusing the threshold each notebook baked into its own CSV).

In [ ]:
def individual_threshold(name, mat):
    t, _ = select_threshold(mat["true_label"], mat[f"{name}_prob"])
    return t

indiv_thresholds = {n: individual_threshold(n, val_mat) for n in MODEL_NAMES}
print("Per-model thresholds selected the same way as the ensemble (validation):", indiv_thresholds)

## 7. TTA note
All three source notebooks already run their own TTA internally (3-view:
identity + horizontal flip + vertical flip) before saving
`probability_malignant` — this is baked into every probability already
loaded above. There is no separate raw-vs-TTA pair of files to compare, so
a distinct "+TTA" ensemble row is not reported; TTA is already reflected in
every row of the table below by construction.

In [ ]:
def final_table_row(name, test_prob, threshold):
    return {"Model": name, **compute_metrics(test_mat["true_label"], test_prob, threshold)}

rows = []
for n in MODEL_NAMES:
    rows.append(final_table_row(n, test_mat[f"{n}_prob"], indiv_thresholds[n]))

test_equal_prob = ensemble_prob(test_mat, equal_weights)
eq_thresh, _ = select_threshold(y_val, ensemble_prob(val_mat, equal_weights))
rows.append(final_table_row("Equal-weight ensemble", test_equal_prob, eq_thresh))

test_opt_prob = ensemble_prob(test_mat, optimal_weights)
rows.append(final_table_row("Optimized weighted ensemble", test_opt_prob, best_threshold))

## 8. Final test-set evaluation — run exactly once, after every choice above is frozen

In [ ]:
comparison_table = pd.DataFrame(rows)[
    ["Model", "roc_auc", "pr_auc", "accuracy", "balanced_accuracy",
     "malignant_recall_sensitivity", "specificity", "malignant_f1", "macro_f1",
     "micro_f1", "micro_recall", "FN", "threshold"]
]
comparison_table.columns = [
    "Model", "ROC-AUC", "PR-AUC", "Accuracy", "Balanced Accuracy", "Malignant Recall",
    "Specificity", "Malignant F1", "Macro F1", "Micro F1 (suppl.)", "Micro Recall (suppl.)",
    "FN", "Threshold",
]
print(comparison_table.round(4).to_string(index=False))
comparison_table.to_csv(OUTPUT_DIR / "final_comparison_table.csv", index=False)

# Confusion matrices, ROC, PR curves for each row
for r in rows:
    prob = {n: test_mat[f"{n}_prob"] for n in MODEL_NAMES}.get(r["Model"])
    if prob is None:
        prob = test_opt_prob if r["Model"] == "Optimized weighted ensemble" else test_equal_prob
    y_pred = (prob >= r["threshold"]).astype(int)
    cm = confusion_matrix(test_mat["true_label"], y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(4, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Benign", "Malignant"], yticklabels=["Benign", "Malignant"], ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(r["Model"])
    fig.savefig(OUTPUT_DIR / "plots" / f"cm_{r['Model'].replace(' ', '_')}.png", bbox_inches="tight", dpi=150)
    plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 5))
for n in MODEL_NAMES:
    fpr, tpr, _ = roc_curve(test_mat["true_label"], test_mat[f"{n}_prob"])
    ax.plot(fpr, tpr, label=f"{n} (AUC={roc_auc_score(test_mat['true_label'], test_mat[f'{n}_prob']):.3f})")
fpr, tpr, _ = roc_curve(test_mat["true_label"], test_opt_prob)
ax.plot(fpr, tpr, "k--", label=f"Optimized ensemble (AUC={roc_auc_score(test_mat['true_label'], test_opt_prob):.3f})")
ax.plot([0, 1], [0, 1], color="gray", linestyle=":")
ax.set_xlabel("FPR"); ax.set_ylabel("TPR"); ax.legend(); ax.set_title("ROC — test set")
fig.savefig(OUTPUT_DIR / "plots" / "roc_curves.png", bbox_inches="tight", dpi=150)
plt.close(fig)

fig, ax = plt.subplots(figsize=(6, 5))
for n in MODEL_NAMES:
    prec, rec, _ = precision_recall_curve(test_mat["true_label"], test_mat[f"{n}_prob"])
    ax.plot(rec, prec, label=f"{n} (AP={average_precision_score(test_mat['true_label'], test_mat[f'{n}_prob']):.3f})")
prec, rec, _ = precision_recall_curve(test_mat["true_label"], test_opt_prob)
ax.plot(rec, prec, "k--", label=f"Optimized ensemble (AP={average_precision_score(test_mat['true_label'], test_opt_prob):.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.legend(); ax.set_title("PR curve — test set")
fig.savefig(OUTPUT_DIR / "plots" / "pr_curves.png", bbox_inches="tight", dpi=150)
plt.close(fig)

## 9. Error analysis (Step 9)

In [ ]:
err = test_mat.copy()
for n in MODEL_NAMES:
    err[f"{n}_pred"] = (err[f"{n}_prob"] >= indiv_thresholds[n]).astype(int)
err["ensemble_prob"] = test_opt_prob
err["ensemble_pred"] = (err["ensemble_prob"] >= best_threshold).astype(int)

false_negatives = err[(err["true_label"] == 1) & (err["ensemble_pred"] == 0)]
false_positives = err[(err["true_label"] == 0) & (err["ensemble_pred"] == 1)]
all_agree = err[err[[f"{n}_pred" for n in MODEL_NAMES]].nunique(axis=1) == 1]
strong_disagree = err[err[[f"{n}_pred" for n in MODEL_NAMES]].nunique(axis=1) > 1]

best_single = comparison_table.iloc[
    comparison_table.loc[comparison_table["Model"] != "Optimized weighted ensemble", "Malignant Recall"].idxmax()
]["Model"]
corrected_by_ensemble = err[(err[f"{best_single}_pred"] != err["true_label"]) & (err["ensemble_pred"] == err["true_label"])]
broken_by_ensemble = err[(err[f"{best_single}_pred"] == err["true_label"]) & (err["ensemble_pred"] != err["true_label"])]

print(f"Test-set false negatives (ensemble): {len(false_negatives)}")
print(f"Test-set false positives (ensemble): {len(false_positives)}")
print(f"All-3-agree cases: {len(all_agree)} / {len(err)}")
print(f"Strong disagreement cases: {len(strong_disagree)}")
print(f"Corrected by ensemble vs best single model ({best_single}): {len(corrected_by_ensemble)}")
print(f"Broken by ensemble vs best single model ({best_single}): {len(broken_by_ensemble)}")

err.to_csv(OUTPUT_DIR / "predictions" / "ensemble_test_predictions_full.csv")
strong_disagree.to_csv(OUTPUT_DIR / "predictions" / "disagreement_cases.csv")

## 10. Final interpretation — answer the 10 questions directly from the table

In [ ]:
def answer_final_questions(table: pd.DataFrame):
    indiv = table[~table["Model"].str.contains("ensemble", case=False)]
    best_ens = table[table["Model"] == "Optimized weighted ensemble"].iloc[0]
    best_indiv_row = indiv.loc[indiv["Malignant Recall"].idxmax()]
    print("1. Highest Malignant Recall (individual):", best_indiv_row["Model"], best_indiv_row["Malignant Recall"])
    print("2. Highest Balanced Accuracy:", table.loc[table["Balanced Accuracy"].idxmax(), "Model"])
    print("3. Highest PR-AUC:", table.loc[table["PR-AUC"].idxmax(), "Model"])
    print("4. Best Macro F1:", table.loc[table["Macro F1"].idxmax(), "Model"])
    print("5. Fewest False Negatives:", table.loc[table["FN"].idxmin(), "Model"])
    best_indiv_by_recall = indiv["Malignant Recall"].max()
    print("6. Ensemble improves Malignant Recall vs best individual?",
          best_ens["Malignant Recall"] > best_indiv_by_recall,
          f"({best_ens['Malignant Recall']:.4f} vs {best_indiv_by_recall:.4f})")
    print("7. Ensemble reduces FN vs best individual (by FN count)?",
          best_ens["FN"] < indiv["FN"].min(), f"({best_ens['FN']} vs {indiv['FN'].min()})")
    print("8. Ensemble specificity acceptable (>= floor)?", best_ens["Specificity"] >= MIN_SPECIFICITY,
          f"({best_ens['Specificity']:.4f})")
    print("9. Ensemble improves Balanced Accuracy vs best individual?",
          best_ens["Balanced Accuracy"] > indiv["Balanced Accuracy"].max())
    print("10. All numbers above are from the untouched test set, evaluated once, after weights "
          "and threshold were frozen on validation — so yes, they're the real, unbiased test-set result.")

answer_final_questions(comparison_table)

## 11. Save reproducible ensemble config

In [ ]:
config = {
    "model_names": MODEL_NAMES,
    "model_prediction_dirs": {k: str(v) for k, v in MODEL_DIRS.items()},
    "ensemble_strategy": "probability-weighted soft voting, weights optimized on validation set "
                          "(Nelder-Mead, multi-start, objective = 0.45*sensitivity + 0.45*balanced_accuracy + 0.10*PR-AUC)",
    "optimized_weights": weights_dict,
    "min_specificity_floor_used_for_threshold_selection": MIN_SPECIFICITY,
    "selected_threshold": best_threshold,
    "per_model_thresholds_matched_methodology": indiv_thresholds,
    "positive_class": "Malignant (label=1)",
    "validation_metrics_at_selected_operating_point": compute_metrics(y_val, val_ensemble_prob_opt, best_threshold),
    "test_metrics_final": compute_metrics(test_mat["true_label"], test_opt_prob, best_threshold),
    "note": "Weights and threshold were selected using only the validation split; "
            "the test split above was scored exactly once, after freezing both.",
}
with open(OUTPUT_DIR / "ensemble_config.json", "w") as f:
    json.dump(config, f, indent=2, default=str)

print("\nSaved:", OUTPUT_DIR / "ensemble_config.json")
print("Saved:", OUTPUT_DIR / "final_comparison_table.csv")